# 10. Propagação da riqueza
Desenvolvimento da propagar_riqueza, que leva a riqueza de um período para o seguinte. Requisito F10.

In [1]:
import sys, os, tempfile
_cwd = os.getcwd()
RAIZ = os.path.dirname(_cwd) if os.path.basename(_cwd) == 'tests' else _cwd
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)
import numpy as np

## Desenvolvimento

Escrita e testada aqui; a versão final está em app/nucleo.py.

In [2]:
def propagar_riqueza(w0, theta, alpha, R, rf):
    """Simula pra frente (Etapas 5 e 6). Em cada periodo consome theta*W,
    investe o que sobrou no alpha otimo e calcula a riqueza do periodo seguinte,
    W = poupanca * R_p. (F9, F10)

    Recebe: w0 (riqueza inicial), theta (as fracoes de consumo, T+1 valores),
    alpha (a carteira otima), R (os retornos brutos sorteados, um por periodo
    e por caminho) e rf (o fator livre de risco bruto).

    Devolve um dicionario com W (riqueza), c (consumo) e S (poupanca), cada um
    com uma linha por caminho e uma coluna por periodo.
    """
    R = np.asarray(R, dtype=float)
    alpha = np.asarray(alpha, dtype=float)
    theta = np.asarray(theta, dtype=float)
    n_paths, T, _ = R.shape
    W = np.empty((n_paths, T + 1))
    c = np.empty((n_paths, T + 1))
    S = np.empty((n_paths, T + 1))
    W[:, 0] = w0
    for t in range(T):
        c[:, t] = theta[t] * W[:, t]
        S[:, t] = W[:, t] - c[:, t]
        R_p = rf + (R[:, t, :] - rf) @ alpha           # (n_paths,)
        W[:, t + 1] = S[:, t] * R_p
    # Condição terminal: consome toda a riqueza (theta_T = 1).
    c[:, T] = theta[T] * W[:, T]
    S[:, T] = W[:, T] - c[:, T]
    return {"W": W, "c": c, "S": S}

**Teste**: a riqueza do período seguinte é a poupança vezes o retorno da carteira, e a riqueza nunca fica negativa.

In [3]:
rng = np.random.default_rng(1)
rf_g = 1.003
T = 6
theta = np.linspace(0.1, 1.0, T+1)
alpha = np.array([0.5])
Rp = np.maximum(1+rng.normal(0.01,0.05,(5,T,1)),0)
out = propagar_riqueza(1.0, theta, alpha, Rp, rf_g)

print('W:'); print(out['W'])

W:
[[1.         0.91362564 0.70374787 0.42848151 0.18778822 0.05797777
  0.00885024]
 [1.         0.89376855 0.68442201 0.41706527 0.19027935 0.05749541
  0.00879825]
 [1.         0.88927978 0.66857872 0.39891967 0.18336822 0.05542266
  0.00830665]
 [1.         0.88825706 0.66623956 0.40242344 0.18101991 0.05641585
  0.00873036]
 [1.         0.84484884 0.60783157 0.36547601 0.16379734 0.04972106
  0.00754716]]


In [4]:
for t in range(T):
    assert np.allclose(out['W'][:,t+1], out['S'][:,t]*(rf_g+(Rp[:,t,:]-rf_g)@alpha))
assert np.all(out['W']>=0)